# **Step 1.** Install Python Packages (Unsloth + Colab T4)
Installs Unsloth and compatible dependencies for Google Colab T4 GPU.
*(Uses PyTorch's native SDPA kernels; avoids manual xformers compilation failures).*

In [ ]:
# Install Unsloth & required fine-tuning dependencies for Google Colab T4 GPU
# Note: xformers is not manually installed here to avoid setup.py compilation errors on Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub


# **Step 2.** Import Python Packages

In [ ]:
import os
import json
import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import notebook_login
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel


# **Step 3.** (Optional) Login to Hugging Face
Run this cell if you plan to push your fine-tuned model or dataset to your Hugging Face account.

In [ ]:
# Authenticate with your Hugging Face account (requires a WRITE token)
notebook_login()


# **Step 4.** Load or Generate Diverse ADHIKAAR Dataset
Checks if `train.jsonl` is uploaded. If not, automatically generates 1,000 diverse synthetic samples across 15 Indian states directly in Colab!

In [ ]:
# Check if train.jsonl is uploaded; if not, generate 1000 diverse synthetic samples on the fly
train_path = "train.jsonl"
val_path = "val.jsonl"

if not os.path.exists(train_path):
    for alt in ["ml_training/data/train.jsonl", "/content/train.jsonl"]:
        if os.path.exists(alt):
            train_path = alt
            val_path = alt.replace("train.jsonl", "val.jsonl")
            break

if not os.path.exists(train_path):
    print("train.jsonl not found. Generating diverse ADHIKAAR dataset (1000 train / 200 val)...")
    import random
    FIRST_NAMES = ["Ramesh", "Suresh", "Murugan", "Kavitha", "Priya", "Ananya", "Ramu", "Venkatesh", "Lakshmi", "Rajesh", "Manish", "Sunita", "Deepak", "Pooja", "Selvakumar", "Subba Rao", "Gurpreet", "Sita", "Balaji", "Bhavani"]
    LAST_NAMES = ["Kumar", "Patel", "Singh", "Sharma", "Devi", "Reddy", "Rao", "Gowda", "Nadar", "Pillai", "Yadav", "Verma", "Gupta", "Kaur", "Joshi", "Patil", "Naidu", "Iyer", "Menon", "Bhatt"]
    STATES = [("Tamil Nadu", ["Thanjavur", "Madurai", "Salem"]), ("Andhra Pradesh", ["Guntur", "Krishna", "Kurnool"]), ("Telangana", ["Warangal", "Nalgonda"]), ("Uttar Pradesh", ["Varanasi", "Meerut"]), ("Karnataka", ["Mandya", "Hassan"]), ("Punjab", ["Ludhiana", "Patiala"])]
    BANKS = [("State Bank of India", "SBIN0"), ("Punjab National Bank", "PUNB0"), ("Canara Bank", "CNRB0"), ("Bank of Baroda", "BARB0"), ("Union Bank of India", "UBIN0"), ("HDFC Bank", "HDFC0")]
    SYS_PROMPT = "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. Your job is to extract ONLY explicitly stated facts from the provided text into the requested JSON schema. If a requested field is not mentioned or cannot be verified from the text, you must set its value to null. Output strictly valid JSON with no markdown fences, no preamble, and no explanation."

    def gen_samples(count):
        rows = []
        for _ in range(count):
            kind = random.choice(["aadhaar", "land", "bank", "income", "marks", "decl"])
            name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
            state, dists = random.choice(STATES)
            dist = random.choice(dists)
            if kind == "aadhaar":
                l4 = f"{random.randint(1000, 9999)}"
                txt = f"GOVT OF INDIA - UIDAI\nName: {name}\nDOB: 12/04/1985\nXXXX XXXX {l4}\nAddress: Dist {dist}, {state}"
                prompt = f"Source Type: aadhaar\nTarget Keys: [\"applicant_name\", \"aadhaar_last4\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"applicant_name": name, "aadhaar_last4": l4}
            elif kind == "land":
                ha = round(random.uniform(0.3, 3.5), 2)
                ht = "individual" if random.random() > 0.15 else "institutional"
                txt = f"REVENUE DEPT - GOVT OF {state.upper()}\nROR-1B PATTA\nKhata: {random.randint(100, 999)}\nPattadar: {name if ht=='individual' else 'Gram Panchayat'}\nCategory: {ht.title()}\nExtent: {ha} Hectares"
                prompt = f"Source Type: land_record\nTarget Keys: [\"land_owned_hectares\", \"holder_type\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"land_owned_hectares": ha, "holder_type": ht}
            elif kind == "bank":
                bname, iprefix = random.choice(BANKS)
                ifsc = f"{iprefix}{random.randint(100000, 999999)}"
                acc = f"{random.randint(10000000000, 999999999999)}"
                txt = f"{bname}\nBranch: {dist}\nIFSC: {ifsc}\nAccount Name: {name}\nAccount No: {acc}"
                prompt = f"Source Type: bank_passbook\nTarget Keys: [\"bank_account_number\", \"bank_ifsc\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"bank_account_number": acc, "bank_ifsc": ifsc}
            elif kind == "income":
                inc = random.choice([48000, 60000, 85000, 120000, 180000, 240000])
                txt = f"TALUK OFFICE {dist.upper()}\nINCOME CERTIFICATE\nAnnual income of {name} is Rs. {inc:,}/- per annum."
                prompt = f"Source Type: income_certificate\nTarget Keys: [\"family_income_annual\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"family_income_annual": float(inc)}
            elif kind == "marks":
                pct = round(random.uniform(55.0, 95.0), 1)
                txt = f"BOARD OF SECONDARY EDUCATION {state.upper()}\nCandidate: {name}\nAggregate Percentage: {pct}%"
                prompt = f"Source Type: marksheet\nTarget Keys: [\"last_exam_percentage\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"last_exam_percentage": pct}
            else:
                if random.random() > 0.4:
                    txt = "Namaste, I am a small cultivator. Nobody in my family pays any income tax."
                    exp = {"is_income_tax_payer": False}
                else:
                    txt = "Hello sir, here are my land records and Aadhaar for the farmer scheme."
                    exp = {"is_income_tax_payer": None}
                prompt = f"Source Type: declaration\nTarget Keys: [\"is_income_tax_payer\"]\n\nCitizen Voice Transcript:\n{txt}"
            rows.append({"messages": [
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": json.dumps(exp)}
            ]})
        return rows

    with open("train.jsonl", "w", encoding="utf-8") as f:
        for item in gen_samples(1000):
            f.write(json.dumps(item) + "\n")
    with open("val.jsonl", "w", encoding="utf-8") as f:
        for item in gen_samples(200):
            f.write(json.dumps(item) + "\n")
    train_path = "train.jsonl"
    val_path = "val.jsonl"

dataset = load_dataset("json", data_files={"train": train_path, "validation": val_path})
print(f"Loaded dataset: {len(dataset['train'])} train rows, {len(dataset['validation'])} val rows.")


# **Step 5.** LoRA & Hardware Settings (Optimized for Colab T4 GPU)
- Base model: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
- Uses 4-bit quantization (~5.5GB VRAM footprint)
- `fp16=True`, `bf16=False` (T4 supports FP16, not BF16)
- LoRA rank 16 on all linear attention + MLP projection layers

In [ ]:
# Hardware & LoRA Hyperparameters for Meta-Llama-3.1-8B-Instruct on T4
config = {
    "huggingface_user": "YOUR_HF_USERNAME",  # Replace with your Hugging Face username
    "model_config": {
        "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", # Llama-3.1-8B 4-bit
        "finetuned_model": "adhikaar-llama3.1-8b-fact-extractor",     # Model output name
        "max_seq_length": 2048,                                      # Max tokens per sample
        "dtype": None,                                               # Auto-detects float16 on T4
        "load_in_4bit": True,                                        # 4-bit QLoRA
    },
    "lora_config": {
        "r": 16,                                                     # LoRA rank
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "lora_alpha": 16,
        "lora_dropout": 0,                                           # 0 is optimized by Unsloth
        "bias": "none",
        "use_gradient_checkpointing": "unsloth",                     # Saves 30% VRAM
        "random_state": 42,
    },
    "training_config": {
        "per_device_train_batch_size": 2,                            # Fits easily in 16GB VRAM
        "gradient_accumulation_steps": 4,                            # Effective batch size = 8
        "warmup_steps": 10,
        "num_train_epochs": 3,                                       # 3 epochs avoids overfitting
        "learning_rate": 2e-4,
        "fp16": True,                                                # T4 supports fp16
        "bf16": False,                                               # T4 does NOT support bf16
        "logging_steps": 10,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 42,
        "output_dir": "outputs",
    }
}
print("Configuration ready for Llama-3.1-8B on T4.")


# **Step 6.** Load Llama-3.1-8B, QLoRA & SFTTrainer

In [ ]:
# 1. Load Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["model_config"]["base_model"],
    max_seq_length=config["model_config"]["max_seq_length"],
    dtype=config["model_config"]["dtype"],
    load_in_4bit=config["model_config"]["load_in_4bit"],
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora_config"]["r"],
    target_modules=config["lora_config"]["target_modules"],
    lora_alpha=config["lora_config"]["lora_alpha"],
    lora_dropout=config["lora_config"]["lora_dropout"],
    bias=config["lora_config"]["bias"],
    use_gradient_checkpointing=config["lora_config"]["use_gradient_checkpointing"],
    random_state=config["lora_config"]["random_state"],
)

# 3. Format dataset using Llama-3.1 native chat template
def format_chatml_prompts(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_data = dataset["train"].map(format_chatml_prompts, batched=True)
val_data = dataset["validation"].map(format_chatml_prompts, batched=True)

# 4. Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=config["model_config"]["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=config["training_config"]["per_device_train_batch_size"],
        gradient_accumulation_steps=config["training_config"]["gradient_accumulation_steps"],
        warmup_steps=config["training_config"]["warmup_steps"],
        num_train_epochs=config["training_config"]["num_train_epochs"],
        learning_rate=config["training_config"]["learning_rate"],
        fp16=config["training_config"]["fp16"],
        bf16=config["training_config"]["bf16"],
        logging_steps=config["training_config"]["logging_steps"],
        optim=config["training_config"]["optim"],
        weight_decay=config["training_config"]["weight_decay"],
        lr_scheduler_type=config["training_config"]["lr_scheduler_type"],
        seed=config["training_config"]["seed"],
        output_dir=config["training_config"]["output_dir"],
        report_to="none",
    ),
)
print("SFTTrainer successfully initialized.")


# **Step 7.** Train Your Fine-Tuned Model
Runs fine-tuning for 3 epochs (~15–20 minutes on Colab T4).

In [ ]:
trainer_stats = trainer.train()


# **Step 8.** Save Trainer Stats & VRAM Metrics

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 2)
total_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 2)
runtime_sec = trainer_stats.metrics.get("train_runtime", 0)

print(f"Training Runtime: {runtime_sec / 60:.2f} minutes")
print(f"Peak VRAM Used: {max_memory} GB / {total_memory} GB ({round(max_memory / total_memory * 100, 1)}%)")

with open("trainer_stats.json", "w") as f:
    json.dump(trainer_stats.metrics, f, indent=4)


# **Step 9.** Save Model & Export Options
- Option A: Save LoRA adapters locally
- Option B: Push LoRA adapters to Hugging Face
- Option C: Push merged 16-bit model for Hugging Face Inference Endpoints / vLLM
- Option D: Push GGUF for local Ollama

In [ ]:
model_name = config["model_config"]["finetuned_model"]
hf_user = config["huggingface_user"]

# 1. Save LoRA adapter locally
model.save_pretrained(f"{model_name}-lora")
tokenizer.save_pretrained(f"{model_name}-lora")
print(f"LoRA adapter saved to {model_name}-lora")

# 2. Push LoRA adapter to Hugging Face Hub (Uncomment if logged in)
# model.push_to_hub(f"{hf_user}/{model_name}", tokenizer=tokenizer)

# 3. Push merged 16-bit model (Uncomment for Hugging Face Inference Endpoints / vLLM)
# model.push_to_hub_merged(f"{hf_user}/{model_name}-merged-16bit", tokenizer=tokenizer, save_method="merged_16bit")

# 4. Push GGUF quantized model (Uncomment for Ollama / llama.cpp)
# model.push_to_hub_gguf(f"{hf_user}/{model_name}-GGUF", tokenizer=tokenizer, quantization_method="q4_k_m")


# **Step 10.** Test Fact Extraction Inference on ADHIKAAR Documents
Validates that the fine-tuned model extracts structured facts into strictly valid JSON.

In [ ]:
# Switch model to fast inference mode
FastLanguageModel.for_inference(model)

# Test sample: simulated Patta Land Record OCR
test_prompt = """Source Type: land_record
Target Keys: ["land_owned_hectares", "holder_type"]

Document OCR Text:
REVENUE DEPARTMENT - GOVT OF TAMIL NADU
EXTRACT OF REGISTER OF RIGHTS (ROR-1B / PATTA)
District: Thanjavur | Taluk: Kumbakonam Rural
Khata Number: 418
Survey/Subdivision Number: 184/2B
Pattadar Name: Murugan K
Holder Category: Individual Cultivator
Classification: Agricultural Wet Land
Total Extent: 1.40 Hectares
Status: Clear Title, Mutation Completed"""

system_prompt = (
    "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. "
    "Your job is to extract ONLY explicitly stated facts from the provided text into the "
    "requested JSON schema. If a requested field is not mentioned or cannot be verified from the "
    "text, you must set its value to null. Output strictly valid JSON with no markdown fences, "
    "no preamble, and no explanation."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=input_ids,
    max_new_tokens=128,
    temperature=0.01,
    use_cache=True,
)

response_text = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
print("--- Raw Model Response ---")
print(response_text)

try:
    parsed = json.loads(response_text)
    print("\n[OK] Parsed Valid JSON:")
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print(f"\n[FAIL] Output is not valid JSON: {e}")
